# Notebook 3: Non-Conservative Forces & Rayleigh Dissipation Function

## Course: AP Physics C & Calculus BC Advanced Mechanics

### Objectives:
1. Understand why standard $L = T - V$ breaks down for non-conservative dissipative forces (friction, drag).
2. Formulate **Rayleigh's Dissipation Function** $D = \frac{1}{2} k (v_x^2 + v_y^2 + v_z^2)$.
3. Apply the **Modified Euler-Lagrange Equation**:
   $$\frac{d}{dt}\left(\frac{\partial L}{\partial \dot{q}_k}\right) - \frac{\partial L}{\partial q_k} + \frac{\partial D}{\partial \dot{q}_k} = 0$$
4. Simulate a **Projectile in Air** with linear viscous drag and compare against ideal vacuum motion.


In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

sp.init_printing()


---
## 1. Symbolic Derivation of Damped Projectile Motion

- **Lagrangian**: $L = \frac{1}{2}m(\dot{x}^2 + \dot{y}^2) - mgy$
- **Rayleigh Dissipation Function**: $D = \frac{1}{2}k(\dot{x}^2 + \dot{y}^2)$
- **Drag Forces**: $F_{fx} = -\frac{\partial D}{\partial \dot{x}} = -k\dot{x}$, $F_{fy} = -\frac{\partial D}{\partial \dot{y}} = -k\dot{y}$


In [ ]:
t = sp.Symbol('t', real=True)
m, g, k = sp.symbols('m g k', positive=True)
x, y = sp.Function('x')(t), sp.Function('y')(t)
vx, vy = sp.diff(x, t), sp.diff(y, t)

# Scalar functions
T = sp.Rational(1, 2) * m * (vx**2 + vy**2)
V = m * g * y
L = T - V
D = sp.Rational(1, 2) * k * (vx**2 + vy**2)

# Modified Euler-Lagrange equations
eq_x = sp.Eq(sp.diff(sp.diff(L, vx), t) - sp.diff(L, x) + sp.diff(D, vx), 0)
eq_y = sp.Eq(sp.diff(sp.diff(L, vy), t) - sp.diff(L, y) + sp.diff(D, vy), 0)

print('Equation of motion for x:')
sp.pprint(eq_x)
print('\nEquation of motion for y:')
sp.pprint(eq_y)


---
## 2. Numerical Simulation: Vacuum vs. Air Resistance Trajectories


In [ ]:
m_val = 1.0     # kg
g_val = 9.81    # m/s^2
k_val = 0.25    # Drag coefficient Ns/m

# Damped Projectile ODE
def damped_projectile_ode(t, state):
    x, y, vx, vy = state
    ax = -(k_val / m_val) * vx
    ay = -g_val - (k_val / m_val) * vy
    return [vx, vy, ax, ay]

# Ideal Vacuum ODE
def ideal_projectile_ode(t, state):
    x, y, vx, vy = state
    return [vx, vy, 0.0, -g_val]

# Initial conditions: Launch at 45 degrees with v0 = 30 m/s
v0 = 30.0
angle = np.radians(45.0)
state0 = [0.0, 0.0, v0 * np.cos(angle), v0 * np.sin(angle)]

t_span = (0, 5.0)
t_eval = np.linspace(0, 5.0, 500)

sol_damped = solve_ivp(damped_projectile_ode, t_span, state0, t_eval=t_eval)
sol_ideal = solve_ivp(ideal_projectile_ode, t_span, state0, t_eval=t_eval)

# Filter for ground level (y >= 0)
mask_damped = sol_damped.y[1] >= 0
mask_ideal = sol_ideal.y[1] >= 0

plt.figure(figsize=(10, 5))
plt.plot(sol_ideal.y[0][mask_ideal], sol_ideal.y[1][mask_ideal], 'r--', label='Vacuum Trajectory (No Drag)')
plt.plot(sol_damped.y[0][mask_damped], sol_damped.y[1][mask_damped], 'b-', label='Rayleigh Dissipation Trajectory (Air Drag)')
plt.title('Projectile Motion: Vacuum vs. Rayleigh Air Drag', fontsize=14)
plt.xlabel('Horizontal Distance x (m)', fontsize=12)
plt.ylabel('Vertical Height y (m)', fontsize=12)
plt.grid(True)
plt.legend(fontsize=11)
plt.savefig('/workspace/scratch/projectile_drag_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Projectile drag plot generated successfully!')
